# 05 - Qwen Default Report Adapter Workflow

# 05 - Qwen 默认报告适配器流程

This Colab runbook is for the new Qwen-backed report task. It assumes YOLO fine-tuning has already finished and focuses on checking/reusing the Qwen LoRA adapter, generating grounded reports, running RAG/LLM evaluation, and launching the public Colab demo.

本 Colab runbook 用于新的 Qwen 版报告任务。它假设 YOLO 微调已经完成，重点是检查/复用 Qwen LoRA adapter、生成可信报告、运行 RAG/LLM 评估，以及启动 Colab 外部可访问的 demo。


## What This Notebook Does / 本 Notebook 做什么

1. Mount Google Drive and configure project paths.  
   挂载 Google Drive，并配置项目路径。
2. Sync the latest GitHub code into the Drive project folder.  
   将 GitHub 最新代码同步到 Drive 项目目录。
3. Install Colab dependencies and the local editable package.  
   安装 Colab 依赖和本地 editable 包。
4. Verify existing YOLO weights and Qwen LoRA adapter artifacts.  
   检查已有 YOLO 权重和 Qwen LoRA adapter 产物。
5. Run reconnect-safe Qwen adapter preparation: complete adapter means skip training.  
   运行可断线重连的 Qwen adapter 准备流程：adapter 完整则跳过训练。
6. Build report context, generate a Qwen report, validate it, and fall back to the deterministic template if required.  
   构建报告 context，生成 Qwen 报告，验证其指标/章节/禁用声明，不合格时自动回退确定性模板。
7. Run RAG/LLM evaluation against the accepted report.  
   对最终被接受的报告运行 RAG/LLM 评估。
8. Save a lightweight evidence backup and launch the public Gradio demo.  
   保存轻量证据备份，并启动外部可访问的 Gradio demo。


## Runtime And Dependencies / 运行环境与依赖

Recommended Colab runtime: GPU, L4 or better.  
推荐 Colab 运行时：GPU，L4 或更高。

Large artifacts stay in Drive and are not pushed to GitHub: dataset, YOLO weights, ONNX export, Qwen LoRA adapter, generated reports, and demo runs.  
大文件保留在 Drive，不进入 GitHub：数据集、YOLO 权重、ONNX、Qwen LoRA adapter、生成报告和 demo 结果。

Important dependencies installed below: `ultralytics`, `transformers`, `accelerate`, `bitsandbytes`, `peft`, `trl`, `gradio`, `fastapi`, `pytest`.  
下面会安装的重要依赖包括：`ultralytics`、`transformers`、`accelerate`、`bitsandbytes`、`peft`、`trl`、`gradio`、`fastapi`、`pytest`。


In [ ]:
# CN: 挂载 Google Drive，并设置统一路径变量。
# EN: Mount Google Drive and define shared path variables.
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/CarDD_YOLO11')
REPO_URL = 'https://github.com/lsdlBlueDragon/ai-powered-vehicle-damage-assessment-pipeline.git'
REPO_CLONE_DIR = Path('/content/ai-powered-vehicle-damage-assessment-pipeline')

YOLO_WEIGHTS = DRIVE_ROOT / 'runs/train/yolo11n_seg/weights/best.pt'
QWEN_ADAPTER_DIR = DRIVE_ROOT / 'llm_adapters/qwen2_5_7b_cardd_report_lora'
CONTEXT_JSON = DRIVE_ROOT / 'reports/qwen7b_report_context.json'
QWEN_REPORT_MD = DRIVE_ROOT / 'reports/qwen7b_final_report.md'
QWEN_REPORT_METADATA = DRIVE_ROOT / 'reports/qwen7b_final_report.metadata.json'
LLM_EVAL_JSON = DRIVE_ROOT / 'reports/llm_eval_summary.json'
LLM_EVAL_MD = DRIVE_ROOT / 'reports/llm_eval_summary.md'
GRADIO_OUTPUT_DIR = DRIVE_ROOT / 'runs/gradio_predict'
SAMPLE_DIR = DRIVE_ROOT / 'demo_upload_samples'

os.environ['DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['YOLO_WEIGHTS'] = str(YOLO_WEIGHTS)
os.environ['QWEN_ADAPTER_DIR'] = str(QWEN_ADAPTER_DIR)
os.environ['CONTEXT_JSON'] = str(CONTEXT_JSON)
os.environ['QWEN_REPORT_MD'] = str(QWEN_REPORT_MD)
os.environ['QWEN_REPORT_METADATA'] = str(QWEN_REPORT_METADATA)
os.environ['LLM_EVAL_JSON'] = str(LLM_EVAL_JSON)
os.environ['LLM_EVAL_MD'] = str(LLM_EVAL_MD)
os.environ['GRADIO_OUTPUT_DIR'] = str(GRADIO_OUTPUT_DIR)
os.environ['SAMPLE_DIR'] = str(SAMPLE_DIR)

print('DRIVE_ROOT =', DRIVE_ROOT)
print('YOLO_WEIGHTS =', YOLO_WEIGHTS)
print('QWEN_ADAPTER_DIR =', QWEN_ADAPTER_DIR)


In [ ]:
# CN: 从 GitHub 同步最新轻量工程代码到 Drive。不会复制数据集或权重。
# EN: Sync latest lightweight engineering code from GitHub to Drive. This does not copy datasets or weights.
%cd /content
!rm -rf ai-powered-vehicle-damage-assessment-pipeline
!git clone https://github.com/lsdlBlueDragon/ai-powered-vehicle-damage-assessment-pipeline.git

# CN: rsync 只覆盖代码、文档、notebook、配置和测试。
# EN: rsync only updates code, docs, notebooks, configs, and tests.
!mkdir -p "$DRIVE_ROOT/src" "$DRIVE_ROOT/scripts" "$DRIVE_ROOT/notebooks" "$DRIVE_ROOT/docs" "$DRIVE_ROOT/configs" "$DRIVE_ROOT/tests"
!rsync -a /content/ai-powered-vehicle-damage-assessment-pipeline/src/ "$DRIVE_ROOT/src/"
!rsync -a /content/ai-powered-vehicle-damage-assessment-pipeline/scripts/ "$DRIVE_ROOT/scripts/"
!rsync -a /content/ai-powered-vehicle-damage-assessment-pipeline/notebooks/ "$DRIVE_ROOT/notebooks/"
!rsync -a /content/ai-powered-vehicle-damage-assessment-pipeline/docs/ "$DRIVE_ROOT/docs/"
!rsync -a /content/ai-powered-vehicle-damage-assessment-pipeline/configs/ "$DRIVE_ROOT/configs/"
!rsync -a /content/ai-powered-vehicle-damage-assessment-pipeline/tests/ "$DRIVE_ROOT/tests/"
!cp /content/ai-powered-vehicle-damage-assessment-pipeline/README.md "$DRIVE_ROOT/README.md"
!cp /content/ai-powered-vehicle-damage-assessment-pipeline/requirements-colab.txt "$DRIVE_ROOT/requirements-colab.txt"
!cp /content/ai-powered-vehicle-damage-assessment-pipeline/pyproject.toml "$DRIVE_ROOT/pyproject.toml"


In [ ]:
# CN: 安装系统依赖、Python 依赖，并以 editable 模式安装本项目。
# EN: Install system dependencies, Python dependencies, and this project in editable mode.
%cd /content/drive/MyDrive/CarDD_YOLO11
!apt-get update -qq
!apt-get install -y -qq git rsync
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements-colab.txt
!python -m pip install -q -e .


In [ ]:
# CN: 检查关键产物是否存在。YOLO 已训练完成，所以 best.pt 应该存在。
# EN: Check required artifacts. YOLO fine-tuning is already complete, so best.pt should exist.
from pathlib import Path

required_files = {
    'YOLO weights / YOLO 权重': YOLO_WEIGHTS,
    'Qwen adapter config / Qwen adapter 配置': QWEN_ADAPTER_DIR / 'adapter_config.json',
    'Qwen adapter weights / Qwen adapter 权重': QWEN_ADAPTER_DIR / 'adapter_model.safetensors',
    'Qwen tokenizer config / Qwen tokenizer 配置': QWEN_ADAPTER_DIR / 'tokenizer_config.json',
    'Qwen tokenizer / Qwen tokenizer': QWEN_ADAPTER_DIR / 'tokenizer.json',
}

for label, path in required_files.items():
    print(f'{label}:', 'OK' if Path(path).exists() else 'MISSING', path)

if not YOLO_WEIGHTS.exists():
    raise FileNotFoundError(f'YOLO weights missing. Run YOLO training notebook first: {YOLO_WEIGHTS}')

adapter_complete = all(Path(path).exists() for label, path in required_files.items() if 'Qwen' in label)
print('\nAdapter complete / Adapter 是否完整:', adapter_complete)
print('If adapter is incomplete, the next Qwen cell will train it. / 如果 adapter 不完整，下一步会训练。')


In [ ]:
# CN: 快速工程验证。若这里只是临时网络/环境问题，可先继续，但建议最终通过。
# EN: Quick engineering verification. If this fails due to transient environment issues, you may continue, but final runs should pass.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python -m compileall src scripts
!python -m pytest tests -q -p no:cacheprovider


In [ ]:
# CN: Qwen adapter 准备步骤。重连安全：完整 adapter 已在 Drive 时会跳过训练。
# EN: Qwen adapter preparation. Reconnect-safe: a complete Drive adapter skips training.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python -m vehicle_damage_pipeline.llm.finetune_report_lora --drive-root "$DRIVE_ROOT" --skip-if-complete --train-examples 1200 --eval-examples 120 --epochs 1 --max-seq-length 1024

# CN: 如果你明确要重新训练 adapter，取消下一行注释并运行。
# EN: If you intentionally want to rebuild the adapter, uncomment and run the next line.
# !python -m vehicle_damage_pipeline.llm.finetune_report_lora --drive-root "$DRIVE_ROOT" --force-retrain --train-examples 1200 --eval-examples 120 --epochs 1 --max-seq-length 1024


In [ ]:
# CN: 构建报告上下文，聚合 YOLO metrics、run summary、demo 输出等证据。
# EN: Build report context from YOLO metrics, run summary, demo outputs, and other evidence.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python -m vehicle_damage_pipeline.report.build_context --drive-root "$DRIVE_ROOT" --output "$CONTEXT_JSON"


In [ ]:
# CN: 默认使用 Qwen LoRA adapter 生成项目级中文报告，并在保存前自动验证输出。
# EN: Generate the project-level Chinese report with Qwen LoRA by default, then validate before saving.
# CN: 如果 Qwen 输出缺少指标、必需章节或包含禁用声明，CLI 会自动回退到确定性模板，并写入 metadata。
# EN: If Qwen misses metrics, required sections, or forbidden-claim checks, the CLI falls back to the deterministic template and records metadata.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python -m vehicle_damage_pipeline.report.generate --context "$CONTEXT_JSON" --output "$QWEN_REPORT_MD" --language Chinese --backend qwen --adapter-dir "$QWEN_ADAPTER_DIR" --metadata-output "$QWEN_REPORT_METADATA"

# CN: 低显存/离线演示时可使用模板后端，不加载 Qwen。
# EN: For low-memory or offline demos, use the template backend without loading Qwen.
# !python -m vehicle_damage_pipeline.report.generate --context "$CONTEXT_JSON" --output "$QWEN_REPORT_MD" --language Chinese --no-qwen --metadata-output "$QWEN_REPORT_METADATA"


In [ ]:
# CN: 检查 Task 1 报告验证结果。若 backend=template 且 fallback_reason=qwen_report_validation_failed，说明 Qwen 输出已被安全回退。
# EN: Inspect Task 1 report validation. If backend=template and fallback_reason=qwen_report_validation_failed, unsafe Qwen output was safely replaced.
import json
from pathlib import Path

metadata = json.loads(Path(QWEN_REPORT_METADATA).read_text(encoding='utf-8'))
print(json.dumps(metadata, ensure_ascii=False, indent=2))

backend = metadata.get('backend')
fallback_reason = metadata.get('fallback_reason')
validation = metadata.get('qwen_validation')

if backend == 'qwen_adapter':
    print('\nQwen report accepted after validation. / Qwen 报告已通过验证并被接受。')
elif fallback_reason == 'qwen_report_validation_failed':
    print('\nQwen report failed validation; deterministic template report was saved instead. / Qwen 报告未通过验证，已保存确定性模板报告。')
elif fallback_reason == 'qwen_adapter_incomplete':
    print('\nQwen adapter incomplete; deterministic template report was saved instead. / Qwen adapter 不完整，已保存确定性模板报告。')
else:
    print('\nReport backend:', backend, 'Fallback reason:', fallback_reason)

if validation:
    print('Validation passed / 验证是否通过:', validation.get('passed'))


In [ ]:
# CN: 运行 RAG/LLM 应用评估：检索覆盖、指标一致性、章节覆盖、禁用夸大声明。
# EN: Run RAG/LLM app evaluation: retrieval coverage, metric grounding, section coverage, forbidden claims.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python -m vehicle_damage_pipeline.eval.run_llm_eval --context "$CONTEXT_JSON" --report "$QWEN_REPORT_MD" --knowledge-root "$DRIVE_ROOT" --output-json "$LLM_EVAL_JSON" --output-markdown "$LLM_EVAL_MD"


In [ ]:
# CN: 展示 Qwen 报告、生成元数据和评估摘要。
# EN: Display the Qwen report, generation metadata, and evaluation summary.
from IPython.display import Markdown, display
import json

for path in [QWEN_REPORT_MD, QWEN_REPORT_METADATA, LLM_EVAL_MD, LLM_EVAL_JSON]:
    print('\n====', path, '====')
    if Path(path).exists():
        if str(path).endswith('.md'):
            display(Markdown(Path(path).read_text(encoding='utf-8')))
        else:
            print(json.dumps(json.loads(Path(path).read_text(encoding='utf-8')), ensure_ascii=False, indent=2)[:4000])
    else:
        print('MISSING / 缺失')


In [ ]:
# CN: 备份本次 Qwen 任务的关键证据到 Drive。不会复制原始数据集或大 checkpoint。
# EN: Back up key evidence for this Qwen task to Drive. Raw datasets and large checkpoints are not copied.
from pathlib import Path
from datetime import datetime
import json, shutil, zipfile

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
backup_dir = DRIVE_ROOT / 'backups' / f'qwen_report_task_{stamp}'
backup_dir.mkdir(parents=True, exist_ok=True)

items = [
    CONTEXT_JSON,
    QWEN_REPORT_MD,
    QWEN_REPORT_METADATA,
    LLM_EVAL_JSON,
    LLM_EVAL_MD,
    DRIVE_ROOT / 'reports/qwen7b_report_sft_metadata.json',
    DRIVE_ROOT / 'reports/yolo11n_seg_test_metrics.json',
    DRIVE_ROOT / 'runs/gradio_predict/prediction_summary.json',
    DRIVE_ROOT / 'notebooks/05_colab_qwen_report_adapter_workflow.ipynb',
]

manifest = {'created_at': datetime.now().isoformat(timespec='seconds'), 'entries': [], 'missing': []}
for src in items:
    src = Path(src)
    if src.exists() and src.is_file():
        dst = backup_dir / src.relative_to(DRIVE_ROOT)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        manifest['entries'].append({'source': str(src), 'backup': str(dst), 'size_bytes': src.stat().st_size})
    else:
        manifest['missing'].append(str(src))

manifest_path = backup_dir / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

zip_path = backup_dir.with_suffix('.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
    for file in backup_dir.rglob('*'):
        if file.is_file():
            zf.write(file, file.relative_to(backup_dir.parent))

print('Backup dir / 备份目录:', backup_dir)
print('Backup zip / 备份压缩包:', zip_path)
print(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
# CN: 启动 Colab 专用公开 Gradio demo。这个 cell 会持续运行，出现 gradio.live 链接后用外部链接打开。
# EN: Launch the Colab public Gradio demo. This cell keeps running; open the gradio.live link when it appears.

# CN: 默认报告后端为 Qwen adapter。首次请求会加载 Qwen，GPU 显存会升高，可能需要等待数分钟。
# EN: The default report backend is the Qwen adapter. The first request loads Qwen, GPU memory rises, and it may take minutes.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python /content/drive/MyDrive/CarDD_YOLO11/scripts/colab_gradio_public_demo.py --weights "$YOLO_WEIGHTS" --output-dir "$GRADIO_OUTPUT_DIR" --sample-dir "$SAMPLE_DIR" --adapter-dir "$QWEN_ADAPTER_DIR" --server-port 7860

# CN: 如果只想演示 YOLO + 模板报告，不加载 Qwen，可改用下一行。
# EN: To demo YOLO + template reports without loading Qwen, use the next line instead.
# !python /content/drive/MyDrive/CarDD_YOLO11/scripts/colab_gradio_public_demo.py --weights "$YOLO_WEIGHTS" --output-dir "$GRADIO_OUTPUT_DIR" --sample-dir "$SAMPLE_DIR" --adapter-dir "$QWEN_ADAPTER_DIR" --server-port 7860 --no-qwen


## Troubleshooting / 常见问题

- If the notebook shows only `http://127.0.0.1:7860`, you are not using the Colab public demo script. Use the final cell above and wait for the `gradio.live` link.  
  如果只看到 `http://127.0.0.1:7860`，说明没有使用 Colab 专用公开 demo 脚本。请运行最后一个 cell，并等待 `gradio.live` 链接。

- If the report metadata shows `fallback_reason=qwen_report_validation_failed`, that is expected protection: Qwen generated text was not accepted because it missed required grounding checks.  
  如果报告 metadata 显示 `fallback_reason=qwen_report_validation_failed`，这是预期的保护机制：Qwen 生成文本没有通过必要的可信性检查，因此未被接受。

- If the Qwen cell loads slowly or GPU memory rises sharply, that is expected when loading Qwen2.5-7B plus LoRA.  
  如果 Qwen cell 很慢或显存明显上升，这是加载 Qwen2.5-7B 和 LoRA 的正常现象。

- If Colab disconnects, rerun cells from mount/config through adapter preparation. A complete Drive adapter will skip training.  
  如果 Colab 断开，重新从挂载/配置运行到 adapter 准备步骤即可。Drive 中 adapter 完整时会跳过训练。

- If you need a lightweight demo, add `--no-qwen` to report/demo commands.  
  如果需要轻量演示，在报告/demo 命令中加入 `--no-qwen`。
